# E2-C1 pre-generation smoke (Colab)

**Purpose:** validate the E2-C1 generation contract end to end with **16 smoke attempts
only** (4 prompts x 2 decodings x 2 generators).

This notebook is an **authorisation gate**, not a corpus builder:

* no full generation - the 20,000-sample run is a separate, separately authorised
  notebook/run;
* no training and no fine-tuning;
* no write under `datasets/`; `corpus_written` stays `False`;
* every model is resolved and licensed **before** any weights are loaded;
* the final cell writes a smoke report to Drive and raises `SystemExit`.

If anything is missing (HF token, Llama gate, prompt/split/licence validation, GPU),
the notebook fails closed and writes a `status: "blocked"` report.

## 1. Configure

In [ ]:
import os
from pathlib import Path

# Paths. The project ZIP lives on Drive; all smoke output is ephemeral under /content.
ZIP_PATH = "/content/drive/MyDrive/StopTheCap.zip"
REPO_DIR = "/content/StopTheCap"
RUN_DIR = "/content/e2c1_smoke"
SMOKE_OUT = "/content/e2c1_smoke/out"
REPORT_PATH = "/content/drive/MyDrive/StopTheCap_reports/e2_c1_generation.json"

# The smoke matrix: 4 prompts x 2 decodings x 2 generators = 16 attempts.
SMOKE_PER_GENERATOR = 4
FULL_RUN_REQUESTED_N = 20000

# Hard authorisation gate. Full generation is a separate, separately authorised
# notebook/run; this notebook must never be able to start it.
AUTHORISED = False

# Never hard-code a token. Colab secrets or the environment provide HF_TOKEN.
HF_TOKEN = os.environ.get("HF_TOKEN")

print("ZIP_PATH            :", ZIP_PATH)
print("REPO_DIR            :", REPO_DIR)
print("RUN_DIR             :", RUN_DIR)
print("SMOKE_OUT           :", SMOKE_OUT)
print("REPORT_PATH         :", REPORT_PATH)
print("SMOKE_PER_GENERATOR :", SMOKE_PER_GENERATOR)
print("AUTHORISED          :", AUTHORISED)
print("HF_TOKEN present    :", bool(HF_TOKEN))

## 2. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted at /content/drive")

## 3. Load the private project ZIP

In [ ]:
# Changing the runtime type restarts the kernel, which wipes the config cell. Restore sane
# defaults so this cell works either way instead of dying with NameError.
if "LOAD_METHOD" not in globals():
    import os
    LOAD_METHOD = "zip"
    ZIP_PATH = "/content/drive/MyDrive/StopTheCap.zip"
    REPO_DIR = "/content/StopTheCap"
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print("NOTE: the config cell had not run in this session; using default paths.")
    print("      Re-run cell 2 if your ZIP lives somewhere else.")

import os, shutil, sys, zipfile
from pathlib import Path

# The smoke run needs code, the E2-C1 manifests and the generation config.
REQUIRED = [
    "ml/__init__.py",
    "ml/ingest/__init__.py",
    "ml/ingest/normalise.py",
    "ml/e2c1/__init__.py",
    "ml/e2c1/constants.py",
    "ml/e2c1/ids.py",
    "ml/e2c1/prompts.py",
    "ml/e2c1/families.py",
    "ml/e2c1/split.py",
    "ml/e2c1/generation.py",
    "ml/e2c1/generate.py",
    "ml/e2c1/build_prompts.py",
    "ml/e2c1/build_splits.py",
    "ml/e2c1/generation_config.json",
    "ml/data/__init__.py",
    "ml/data/dedup.py",
    "ml/data/splits.py",
    "datasets/manifests/e2_c1_prompts.jsonl",
    "datasets/manifests/e2_c1_prompt_audit.json",
    "datasets/manifests/e2_c1_splits.json",
    "datasets/manifests/e2_c1_licences.json",
    "notebooks/colab_generate_e2c1.ipynb",
]
EXTRACT_TMP = "/content/_stc_extract"   # overridden in tests

def _fail(msg):
    raise RuntimeError(msg)

def load_from_zip(zip_path, dest):
    z = Path(zip_path)
    if not z.is_file():
        _fail(
            "ZIP not found: " + str(z) + "\n"
            "Upload your project ZIP to Google Drive, then set ZIP_PATH in the previous cell.\n"
            "Expected form: /content/drive/MyDrive/<name>.zip"
        )
    dest = Path(dest)
    tmp = Path(EXTRACT_TMP)
    for p in (dest, tmp):
        if p.exists():
            shutil.rmtree(p)
    tmp.mkdir(parents=True)

    with zipfile.ZipFile(z) as zf:
        names = zf.namelist()
        print("zip entries:", len(names))
        for n in names[:12]:
            print("   ", n)
        if not names:
            _fail("The ZIP is empty.")
        zf.extractall(tmp)

    entries = [p for p in tmp.iterdir() if p.name != "__MACOSX"]
    # Flatten a single top-level folder so we never end up with StopTheCap/StopTheCap.
    root = entries[0] if len(entries) == 1 and entries[0].is_dir() else tmp
    shutil.move(str(root), str(dest))
    if tmp.exists():
        shutil.rmtree(tmp, ignore_errors=True)
    return dest

if LOAD_METHOD == "zip":
    REPO = load_from_zip(ZIP_PATH, REPO_DIR)
elif LOAD_METHOD == "git":
    _fail("This private notebook loads the Drive ZIP only; set LOAD_METHOD = 'zip'.")
else:
    _fail("Unknown LOAD_METHOD " + repr(LOAD_METHOD) + ". Use 'zip'.")

REPO = Path(REPO).resolve()
missing = [f for f in REQUIRED if not (REPO / f).is_file()]
if missing:
    _fail(
        "Project structure looks wrong; missing: " + ", ".join(missing) + "\n"
        "The ZIP must contain the repository root - the folder that holds ml/, datasets/, notebooks/."
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

all_files = [p for p in REPO.rglob("*") if p.is_file()]
print()
print("project root :", REPO)
print("file count   :", len(all_files))
print("size MB      :", round(sum(p.stat().st_size for p in all_files) / 1e6, 2))
print("required     :", len(REQUIRED), "present")

## 4. Dependencies

In [ ]:
%pip install -q "transformers>=4.45,<5" "accelerate>=0.30" "bitsandbytes>=0.43" sentencepiece protobuf "pydantic>=2.7" "PyYAML>=6.0"

import accelerate
import bitsandbytes
import google.protobuf
import pydantic
import sentencepiece
import transformers
import yaml

VERSIONS = {
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "sentencepiece": getattr(sentencepiece, "__version__", "unknown"),
    "protobuf": google.protobuf.__version__,
    "pydantic": pydantic.__version__,
    "PyYAML": yaml.__version__,
}
for name, version in VERSIONS.items():
    print(f"{name:<14} {version}")

## 5. Hardware pre-flight

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "CUDA is required for the E2-C1 smoke. Choose Runtime > Change runtime type > T4 GPU."
)
props = torch.cuda.get_device_properties(0)
gpu_name = props.name
vram_gb = props.total_memory / 1e9
capability = torch.cuda.get_device_capability(0)
print(f"GPU        : {gpu_name}")
print(f"VRAM       : {vram_gb:.2f} GB")
print(f"capability : sm_{capability[0]}{capability[1]}")
assert props.total_memory >= 14.5e9, (
    f"Need at least 14.5 GB VRAM for 4-bit Mistral + fp16 Llama; found {vram_gb:.2f} GB."
)

GPU_INFO = {
    "name": gpu_name,
    "vram_gb": round(vram_gb, 2),
    "capability": f"sm_{capability[0]}{capability[1]}",
}

from ml.e2c1.generation import config_digest, load_config

CONFIG_PATH = REPO / "ml/e2c1/generation_config.json"
config = load_config(CONFIG_PATH)
CONFIG_DIGEST = config_digest(config)

n_generators = len(config.generators)
n_decodings = len(config.decodings)
n_planned = n_generators * n_decodings * 5000
assert n_planned == 20000, f"expected 20,000 planned samples, got {n_planned}"
print(f"config_digest : {CONFIG_DIGEST}")
print(f"generators    : {n_generators}")
print(f"decodings     : {n_decodings}")
print(f"planned       : {n_generators} x {n_decodings} x 5000 = {n_planned}")

## 6. Licence registry validation

In [ ]:
from ml.e2c1.generation import load_registry, validate_config_against_licences

REGISTRY_PATH = REPO / "datasets/manifests/e2_c1_licences.json"
registry = load_registry(REGISTRY_PATH)
validate_config_against_licences(config, registry)

by_generator = {entry["generator_id"]: entry for entry in registry.get("models", [])}
LICENCE_ROWS = []
print(
    f"{'generator':<26} {'model':<36} {'revision':<42} {'spdx':<12} "
    f"{'status':<34} {'gated':<6} {'licensed':<9} quantisation"
)
for generator in config.generators:
    entry = by_generator[generator.generator_id]
    licence = entry.get("licence", {})
    outputs = entry.get("output_terms", {})
    quantisation = generator.quantisation.model_dump(mode="json")
    outputs_licensed = outputs.get("outputs_licensed")
    print(
        f"{generator.generator_id:<26} {generator.model_id:<36} "
        f"{generator.model_revision:<42} {licence.get('spdx', ''):<12} "
        f"{entry.get('status', ''):<34} {str(entry.get('gated')):<6} "
        f"{str(outputs_licensed):<9} {quantisation.get('method')}"
    )
    assert outputs_licensed is False, (
        f"{generator.generator_id}: outputs_licensed must be False; "
        "the weight licence does not license generated text"
    )
    blob = f"{generator.generator_id} {generator.model_id}".lower()
    if "llama" in blob:
        assert licence.get("spdx") != "Apache-2.0", (
            "Llama must not be described as Apache-2.0"
        )
        assert licence.get("not_apache_2_0") is True, (
            "Llama must record not_apache_2_0: true"
        )
    LICENCE_ROWS.append(
        {
            "generator_id": generator.generator_id,
            "model_id": generator.model_id,
            "model_revision": generator.model_revision,
            "spdx": licence.get("spdx"),
            "status": entry.get("status"),
            "gated": entry.get("gated"),
            "outputs_licensed": outputs_licensed,
            "quantisation": quantisation,
        }
    )
print()
print("licence registry validated:", len(LICENCE_ROWS), "generators")

## 7. Prompt manifest + split validation

In [ ]:
import hashlib
import json

from ml.e2c1.prompts import read_manifest, validate_manifest
from ml.e2c1.split import validate_split_manifest

PROMPTS_PATH = REPO / "datasets/manifests/e2_c1_prompts.jsonl"
SPLITS_PATH = REPO / "datasets/manifests/e2_c1_splits.json"

prompt_rows = read_manifest(PROMPTS_PATH)
assert len(prompt_rows) == 5000, (
    f"prompt manifest has {len(prompt_rows)} rows, expected 5000"
)
manifest_problems = validate_manifest(prompt_rows)
assert not manifest_problems, "prompt manifest violations:\n" + "\n".join(manifest_problems)
PROMPT_MANIFEST_SHA256 = hashlib.sha256(PROMPTS_PATH.read_bytes()).hexdigest()

split = json.loads(SPLITS_PATH.read_text(encoding="utf-8"))
assert split.get("prompt_manifest_sha256") == PROMPT_MANIFEST_SHA256, (
    "split manifest prompt_manifest_sha256 does not match the JSONL bytes"
)
split_problems = validate_split_manifest(split, prompt_rows)
assert not split_problems, "split manifest violations:\n" + "\n".join(split_problems)
assignments = {str(key): str(value) for key, value in split["assignments"].items()}
SPLIT_DIGEST = split["split_digest"]
assert set(split["partitions"]) == {"train", "val", "test_holdout"}, split["partitions"]
assert "test_id" not in set(assignments.values()), "no E2-C1 prompt may map to test_id"

generator_ids = [generator.generator_id for generator in config.generators]
decoding_ids = [decoding.decoding_config_id for decoding in config.decodings]
matrix = split["counts"]["by_partition_generator_decoding_topic"]

print(
    f"{'partition':<13} {'topic':<20} {'generator':<26} {'decoding':<10} count"
)
for partition in split["partitions"]:
    for generator_id in generator_ids:
        for decoding_id in decoding_ids:
            by_topic = matrix[partition][generator_id][decoding_id]
            for topic in sorted(by_topic):
                print(
                    f"{partition:<13} {topic:<20} {generator_id:<26} "
                    f"{decoding_id:<10} {by_topic[topic]}"
                )
            planned = sum(by_topic.values())
            assert planned > 0, (
                f"empty planned generation cell: "
                f"{partition}/{generator_id}/{decoding_id}"
            )
print()
print("prompt_manifest_sha256:", PROMPT_MANIFEST_SHA256)
print("split_digest          :", SPLIT_DIGEST)
print("partitions            :", split["partitions"])

## 8. Credentials and gate check

In [ ]:
import json
import os
import tempfile
from pathlib import Path

from huggingface_hub import HfApi, get_token
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError, RepositoryNotFoundError

token = get_token() or HF_TOKEN
print("HF token present:", bool(token))


def _atomic_write_report(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    data = json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False) + "\n"
    descriptor, temp_name = tempfile.mkstemp(
        prefix=path.name + ".", suffix=".tmp", dir=str(path.parent)
    )
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8") as handle:
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp_name, path)
    except BaseException:
        try:
            os.unlink(temp_name)
        except OSError:
            pass
        raise


def _blocked_report(reason, remedy):
    return {
        "schema_version": "1.0",
        "mode": "smoke",
        "status": "blocked",
        "reason": reason,
        "remedy": remedy,
        "config_digest": CONFIG_DIGEST,
        "gpu": GPU_INFO,
        "versions": VERSIONS,
        "licences": LICENCE_ROWS,
        "records": 0,
        "corpus_written": False,
        "full_run": {"authorised": False, "requested_n": 20000},
        "next_action": (
            "Provide HF_TOKEN and gated access, then re-run this smoke-only notebook."
        ),
    }


if not token:
    payload = _blocked_report(
        "missing_token",
        "Set HF_TOKEN in Colab secrets (Runtime > Secrets) or export it before starting.",
    )
    _atomic_write_report(REPORT_PATH, payload)
    raise RuntimeError("Blocked: HF_TOKEN is missing. " + payload["remedy"])

api = HfApi(token=token)
gate_failures = []
for generator in config.generators:
    try:
        info = api.model_info(generator.model_id, revision=generator.model_revision)
        print(f"resolved {generator.generator_id}: {info.id} @ {info.sha}")
    except (GatedRepoError, RepositoryNotFoundError) as exc:
        gate_failures.append((generator.generator_id, f"{type(exc).__name__}: {exc}"))
        break
    except HfHubHTTPError as exc:
        status_code = getattr(getattr(exc, "response", None), "status_code", None)
        if status_code in (401, 403):
            detail = f"auth failure HTTP {status_code}: {exc}"
        else:
            detail = f"HTTP {status_code}: {exc}"
        gate_failures.append((generator.generator_id, detail))
        break

if gate_failures:
    generator_id, detail = gate_failures[0]
    remedy = (
        f"Accept the gated model terms for {generator_id} on Hugging Face and use a token "
        "with read access to gated repositories. Nothing was generated."
    )
    payload = _blocked_report(f"model_resolution_failed:{generator_id}", remedy)
    payload["detail"] = detail
    _atomic_write_report(REPORT_PATH, payload)
    raise RuntimeError(f"Blocked before any model load: {detail}. {remedy}")

print("both models resolved; no weights loaded yet")

## 9. Smoke generation

In [ ]:
import json
import statistics
import subprocess
import sys
import time
from collections import Counter
from pathlib import Path

# The CLI picks 4 prompts per generator deterministically (2 train / 1 val / 1 test_holdout
# by sha256("e2c1-smoke-v1|" + prompt_id) rank). We run one generator at a time.
SMOKE_OUT_PATH = Path(SMOKE_OUT)
SMOKE_REPORTS = []
SMOKE_PROMPT_IDS = set()

for generator in config.generators:
    generator_out = SMOKE_OUT_PATH / generator.generator_id
    generator_out.mkdir(parents=True, exist_ok=True)
    report_file = generator_out / "report.json"
    command = [
        sys.executable, "-m", "ml.e2c1.generate",
        "--config", str(CONFIG_PATH),
        "--prompts", str(PROMPTS_PATH),
        "--splits", str(SPLITS_PATH),
        "--out", str(generator_out),
        "--smoke", str(SMOKE_PER_GENERATOR),
        "--generator", generator.generator_id,
        "--report", str(report_file),
    ]
    print("running:", " ".join(command))
    began = time.perf_counter()
    completed = subprocess.run(command, cwd=str(REPO), text=True, capture_output=True)
    wall_seconds = time.perf_counter() - began
    if completed.stdout:
        print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(
            f"smoke subprocess failed for {generator.generator_id} "
            f"(exit {completed.returncode}); no full generation was attempted"
        )
    report = json.loads(report_file.read_text(encoding="utf-8"))
    report["_wall_seconds"] = wall_seconds
    SMOKE_REPORTS.append(report)
    SMOKE_PROMPT_IDS.update(report["smoke"]["prompt_ids"])
    print(
        f"{generator.generator_id}: records={report['n_records']} "
        f"failures={report['n_failures']} attempts={report['attempts']}"
    )

attempts = sum(int(report["attempts"]) for report in SMOKE_REPORTS)
records = sum(int(report["n_records"]) for report in SMOKE_REPORTS)
failures = sum(int(report["n_failures"]) for report in SMOKE_REPORTS)
expected_attempts = SMOKE_PER_GENERATOR * len(config.decodings) * len(config.generators)
assert attempts == expected_attempts, f"attempts {attempts} != {expected_attempts}"
assert attempts == 4 * 2 * 2, (
    "expected 4 prompts x 2 decodings x 2 generators = 16 attempts"
)
assert records + failures == attempts, (
    f"{records} successes + {failures} failures != {attempts} attempts"
)
assert len(SMOKE_PROMPT_IDS) == SMOKE_PER_GENERATOR, SMOKE_PROMPT_IDS
coverage = Counter(assignments[prompt_id] for prompt_id in SMOKE_PROMPT_IDS)
assert coverage["train"] == 2, coverage
assert coverage["val"] == 1, coverage
assert coverage["test_holdout"] == 1, coverage

per_sample = [
    float(value)
    for report in SMOKE_REPORTS
    for value in report["timings"]["per_sample_s"]
]
median_seconds = statistics.median(per_sample) if per_sample else None
eta_full_hours = (
    median_seconds * FULL_RUN_REQUESTED_N / 3600.0 if median_seconds else None
)

print()
print(f"attempts                  : {attempts}")
print(f"successes                 : {records}")
print(f"failures                  : {failures}")
print(f"prompt ids                : {sorted(SMOKE_PROMPT_IDS)}")
print(f"per-sample seconds        : {[round(value, 3) for value in per_sample]}")
print(f"median per-sample seconds : {median_seconds}")
print(f"eta_full_hours            : {eta_full_hours}")

## 10. Report and hard stop

In [ ]:
import json
import os
import statistics
import tempfile
from datetime import datetime, timezone
from pathlib import Path

# This notebook can only ever write a smoke report. Full generation is a separate,
# separately authorised notebook/run.
assert AUTHORISED is False, (
    "Full generation is a separate, separately authorised notebook/run; "
    "this notebook is smoke-only and must not start a corpus."
)

report_path = Path(REPORT_PATH)
report_path.parent.mkdir(parents=True, exist_ok=True)

smoke_attempts = sum(int(report["attempts"]) for report in SMOKE_REPORTS)
smoke_records = sum(int(report["n_records"]) for report in SMOKE_REPORTS)
smoke_failures = sum(int(report["n_failures"]) for report in SMOKE_REPORTS)
if smoke_records + smoke_failures != smoke_attempts:
    raise RuntimeError("smoke accounting mismatch: successes + failures != attempts")
per_sample = [
    float(value)
    for report in SMOKE_REPORTS
    for value in report["timings"]["per_sample_s"]
]
median_seconds = statistics.median(per_sample) if per_sample else None
eta_full_hours = median_seconds * 20000 / 3600.0 if median_seconds else None
failure_records = [
    failure
    for report in SMOKE_REPORTS
    for failure in report["smoke"]["failures"]
]

payload = {
    "schema_version": "1.0",
    "mode": "smoke",
    "status": "ok",
    "generated_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "config_digest": CONFIG_DIGEST,
    "prompt_manifest_sha256": PROMPT_MANIFEST_SHA256,
    "split_digest": SPLIT_DIGEST,
    "gpu": GPU_INFO,
    "versions": VERSIONS,
    "licences": LICENCE_ROWS,
    "quantisation": {
        generator.generator_id: generator.quantisation.model_dump(mode="json")
        for generator in config.generators
    },
    "smoke": {
        "n_prompts_per_generator": SMOKE_PER_GENERATOR,
        "n_records": smoke_records,
        "n_attempts": smoke_attempts,
        "prompt_ids": sorted(SMOKE_PROMPT_IDS),
        "failures": failure_records,
    },
    "timings": {
        "per_sample_s": per_sample,
        "median_s": median_seconds,
        "eta_full_hours": eta_full_hours,
    },
    "full_run": {"authorised": False, "requested_n": 20000},
    "corpus_written": False,
    "next_action": (
        "Review the smoke report, licence table and eta_full_hours, then authorise "
        "a separate full-generation notebook/run explicitly."
    ),
}

data = json.dumps(payload, indent=2, sort_keys=True, ensure_ascii=False) + "\n"
descriptor, temp_name = tempfile.mkstemp(
    prefix=report_path.name + ".", suffix=".tmp", dir=str(report_path.parent)
)
try:
    with os.fdopen(descriptor, "w", encoding="utf-8") as handle:
        handle.write(data)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temp_name, report_path)
except BaseException:
    try:
        os.unlink(temp_name)
    except OSError:
        pass
    raise

print("=" * 72)
print("E2-C1 SMOKE REPORT WRITTEN")
print(f"  report                : {report_path}")
print("  mode                  : smoke")
print(f"  attempts              : {smoke_attempts}")
print(f"  successes / failures  : {smoke_records} / {smoke_failures}")
print(f"  eta_full_hours        : {eta_full_hours}")
print("  corpus_written        : False")
print("=" * 72)
raise SystemExit("AUTHORISATION REQUIRED - smoke only")